# EYES-DEFY-ANEMIA -- Phase 4 Classification -- new_way/ 3-fold retraining (kfold3)

Fixed-hyperparameter 3-fold cross-validation retraining of the **6 best combos** from the
`new_way/` 16-combo Optuna sweep (ranked by validation F1 in
`Output/version1/compare/new_way_model_comparison.xlsx`):

| Model | Tissue | Original F1 |
|---|---|---|
| ConvNeXt-Base | palpebral | 0.9333 |
| ConvNeXt-Large | palpebral | 0.9333 |
| CoAtNet-3 | palpebral | 0.8966 |
| EfficientNet-B3 | forniceal_palpebral | 0.8966 |
| MaxViT-Tiny | palpebral | 0.8750 |
| RegNetY-16GF | palpebral | 0.8667 |

**No Optuna in this notebook at all.** Each combo's `learning_rate`/`weight_decay`/`dropout_rate`
are fixed at its own already-found best trial's values (read from its real
`Output/version1/logs/*_study_summary.json` by `new_way/kfold/_generate_scripts.py`, not
hand-copied) -- the point of this run is a more robust *measurement* of each combo's real
performance (mean +/- std across 3 independent trainings), not a better hyperparameter search.

**Data:** each fold's TRAIN portion uses the offline, country-stratified label-balanced dataset
(`new_way/Offline_data_augmentation/`) *with* online augmentation (HorizontalFlip/Rotate) still
layered on top -- both, not one instead of the other, per explicit instruction. VAL and the
sealed TEST split are always real, unmodified, unaugmented images.

**Protocol:** 3-fold `StratifiedKFold` on the country+label compound key, over the pooled
train+val patients (test stays sealed) -- one fixed fold partition per tissue type, reused
identically across every model using that tissue type. `ReduceLROnPlateau` (factor=0.5,
patience=5, min_lr=1e-6), gradient clipping (max_norm=1.0), early stopping (patience=15, so the
scheduler gets a real chance to act first), 250-epoch ceiling, batch size 32 -- mirrors
`classification/datapreparepipeline/efficientnet_b0_forniceal_5fold_cv/cv_trainer_engine.py`'s
own fixed-hyperparameter CV numbers exactly (this project's own most directly analogous
precedent). Checkpoints saved fp16 (halves disk footprint across 6 architectures x 3 folds).

Real local timing (RTX 4050): EfficientNet-B3 ~0.94s/epoch, ConvNeXt-Large ~3.37s/epoch --
even a pessimistic worst case (all 18 fits run the full 250 epochs, no early stopping) is
~4.2h, comfortably one Kaggle session; realistically much less once early stopping engages.

`sync_outputs()` runs after every fold-completing training cell, so an interrupted session
still yields a downloadable zip of everything completed so far.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 2296, done.
remote: Counting objects: 100% (1977/1977), done.
remote: Compressing objects: 100% (1523/1523), done.
remote: Total 2296 (delta 598), reused 1783 (delta 440), pack-reused 319 (from 1)
Receiving objects: 100% (2296/2296), 103.21 MiB | 27.75 MiB/s, done.
Resolving deltas: 100% (761/761), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic only -- kept for visibility in the saved run log.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [4]:
# optuna is still required even though this notebook never runs a search --
# datapreparepipeline/trainer_engine.py (reused for ARCHITECTURE_REGISTRY/
# compute_metrics/evaluate) imports it unconditionally at module level.
# timm is required for coatnet_3 (not in torchvision at all).
!pip install -q optuna albumentations timm

## Data

Same Kaggle-dataset copy step as the original `classification-new-way.ipynb` -- verify
`SRC_DIR` against the `/kaggle/input` listing above before running if this is a fresh Kaggle
dataset attachment.

**`new_way/Offline_data_augmentation/` (the balanced train images + manifest.csv, ~4.3MB) is
NOT part of this copy step** -- it's small enough to live in the git repo directly and arrives
via the `git clone` above. The sanity-check cell below asserts it's actually present before any
training starts, so a forgotten `git push` fails loudly here instead of silently deep inside a
training run.

In [5]:
import shutil
from pathlib import Path

# TODO: verify against the /kaggle/input listing cell above before running.
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


In [6]:
# Confirms the offline-balanced dataset arrived via git clone (see markdown above) --
# fails loudly here, not deep inside the first training cell, if it wasn't committed/pushed.
import pandas as pd

manifest_path = Path("classification/new_way/Offline_data_augmentation/manifest.csv")
assert manifest_path.exists(), (
    f"{manifest_path} not found. Offline_data_augmentation/ must be committed and pushed "
    "to GitHub before this notebook can clone it -- it is NOT part of the Kaggle-uploaded "
    "processed-dataset."
)
manifest = pd.read_csv(manifest_path)
print(f"manifest.csv: {len(manifest)} rows")
print(manifest.groupby(["tissue_type", "country", "anemic_label"]).size())

manifest.csv: 554 rows
tissue_type          country  anemic_label
forniceal_palpebral  India    0.0             57
                              1.0             57
                     Italy    0.0             79
                              1.0             79
palpebral            India    0.0             57
                              1.0             57
                     Italy    0.0             84
                              1.0             84
dtype: int64


## Registry + fold-building sanity check

Confirms the 6 target architectures actually build/forward-pass, and independently re-derives
each tissue type's 3-fold split (patient counts, no overlap between folds' validation sets) --
before any real training starts.

In [7]:
import sys
sys.path.insert(0, "classification/datapreparepipeline")
sys.path.insert(0, "classification/new_way")
sys.path.insert(0, "classification/new_way/kfold")

import torch
from trainer_engine import ARCHITECTURE_REGISTRY, DEVICE

TARGET_ARCHS = ["efficientnet_b3", "maxvit_t", "regnet_y_16gf", "convnext_base", "coatnet_3", "convnext_large"]

for arch in TARGET_ARCHS:
    cfg = ARCHITECTURE_REGISTRY[arch]
    model = cfg["build_fn"](0.2).to(DEVICE)
    x = torch.randn(2, 3, cfg["input_size"], cfg["input_size"]).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    assert out.shape == (2, 1), f"{arch}: bad output shape {out.shape}"
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{arch:<20} OK  out={tuple(out.shape)}  trainable_params={n_trainable}")
    del model
print("\nAll 6 target architectures registered and working.")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 136MB/s]


efficientnet_b3      OK  out=(2, 1)  trainable_params=1537
Downloading: "https://download.pytorch.org/models/maxvit_t-bc5ab103.pth" to /root/.cache/torch/hub/checkpoints/maxvit_t-bc5ab103.pth


100%|██████████| 119M/119M [00:06<00:00, 18.4MB/s]


maxvit_t             OK  out=(2, 1)  trainable_params=513
Downloading: "https://download.pytorch.org/models/regnet_y_16gf-9e6ed7dd.pth" to /root/.cache/torch/hub/checkpoints/regnet_y_16gf-9e6ed7dd.pth


...

convnext_base        OK  out=(2, 1)  trainable_params=1025


model.safetensors:   0%|          | 0.00/727M [00:00<?, ?B/s]

coatnet_3            OK  out=(2, 1)  trainable_params=1537
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 199MB/s]


convnext_large       OK  out=(2, 1)  trainable_params=1537

All 6 target architectures registered and working.


In [8]:
import kfold_engine as ke

for tissue in ["palpebral", "forniceal_palpebral"]:
    pool = ke.load_pool_df(tissue)
    folds = ke.build_folds(pool)
    print(f"=== {tissue}: pool={len(pool)} patients, {len(folds)} folds ===")
    total, seen_val = 0, set()
    for i, (tr, va) in enumerate(folds, 1):
        overlap = seen_val & set(va)
        assert not overlap, f"fold {i} overlaps a previous fold's val set: {overlap}"
        seen_val |= set(va)
        total += len(va)
        print(f"  fold {i}: train={len(tr)} val={len(va)}")
    assert total == len(pool), f"fold val sizes sum to {total}, expected {len(pool)}"
print("\nFold geometry OK for both tissue types.")

=== palpebral: pool=184 patients, 3 folds ===
  fold 1: train=122 val=62
  fold 2: train=123 val=61
  fold 3: train=123 val=61
=== forniceal_palpebral: pool=178 patients, 3 folds ===
  fold 1: train=118 val=60
  fold 2: train=119 val=59
  fold 3: train=119 val=59

Fold geometry OK for both tissue types.


## Output syncing

In [9]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate new_way/Output/version2/{checkpoints,logs,plots}/ into a
    single top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/new_way_kfold3_results.zip. Called after EVERY training
    cell -- 6 models x 3 folds is a long unattended run, so whatever
    completed so far must always be downloadable."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/new_way/Output/version2") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/new_way_kfold3_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

[sync_outputs] 0 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/new_way_kfold3_results.zip


## Training -- 6 combos x 3 folds = 18 fits, cheapest architecture first

Each script's `model_name` carries a `_kfold3` suffix so these results never collide with the
`version1` Optuna-sweep results under the un-suffixed name. Real local per-epoch timing
(RTX 4050): EfficientNet-B3 ~0.94s, ConvNeXt-Large ~3.37s -- ordering cheapest-first means an
interrupted session still banks the fastest, most numerous results first.

In [10]:
# new_way kfold3 1/6 -- efficientnet_b3, forniceal_palpebral (lightest)
!python classification/new_way/kfold/train_kfold_efficientnet_b3_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Model: efficientnet_b3_forniceal_palpebral_new_way_kfold3 (fixed-hyperparameter 3-fold CV, no Optuna)
arch_name=efficientnet_b3 tissue_type=forniceal_palpebral
learning_rate=0.006358358856676255 weight_decay=0.000133112160807369 dropout_rate=0.5
MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=15 BATCH_SIZE=32
Pool (train+val): 178 patients -> 3 folds

efficientnet_b3_forniceal_palpebral_new_way_kfold3 | Fold 1/3 -- train=118 val=60
[efficientnet_b3_forniceal_palpebral_new_way_kfold3 | fold 1] train n=180 (pos=90 neg=90 pos_weight=1.000) val n=60
[efficientnet_b3_forniceal_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.0000 -> saved efficientnet_b3_forniceal_palpebral_new_way_kfold3_fold1_best.pth
[efficientnet_b3_forniceal_palpebral_new_way_kfold3 | fold 1] Epoch   1/250 - train_loss=0.6791 val_loss=0.6935 val_f1=0.0000 lr=6.36e-03
[efficientnet_b3_forniceal_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.2581 -> saved efficientnet_b3_forniceal_palpebral_new_way_kf

In [11]:
# new_way kfold3 2/6 -- maxvit_t, palpebral
!python classification/new_way/kfold/train_kfold_maxvit_t_palpebral_new_way.py
sync_outputs()

Using device: cuda
Model: maxvit_t_palpebral_new_way_kfold3 (fixed-hyperparameter 3-fold CV, no Optuna)
arch_name=maxvit_t tissue_type=palpebral
learning_rate=0.0008179499475211679 weight_decay=3.752055855124284e-05 dropout_rate=0.2
MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=15 BATCH_SIZE=32
Pool (train+val): 184 patients -> 3 folds

maxvit_t_palpebral_new_way_kfold3 | Fold 1/3 -- train=122 val=62
[maxvit_t_palpebral_new_way_kfold3 | fold 1] train n=186 (pos=92 neg=94 pos_weight=1.022) val n=62
[maxvit_t_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.2162 -> saved maxvit_t_palpebral_new_way_kfold3_fold1_best.pth
[maxvit_t_palpebral_new_way_kfold3 | fold 1] Epoch   1/250 - train_loss=0.6810 val_loss=0.7071 val_f1=0.2162 lr=8.18e-04
[maxvit_t_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.2439 -> saved maxvit_t_palpebral_new_way_kfold3_fold1_best.pth
[maxvit_t_palpebral_new_way_kfold3 | fold 1] Epoch   2/250 - train_loss=0.6777 val_loss=0.7077 val_f1=0.2439 lr=8.18e-04
[maxvit_t_

In [12]:
# new_way kfold3 3/6 -- regnet_y_16gf, palpebral
!python classification/new_way/kfold/train_kfold_regnet_y_16gf_palpebral_new_way.py
sync_outputs()

Using device: cuda
Model: regnet_y_16gf_palpebral_new_way_kfold3 (fixed-hyperparameter 3-fold CV, no Optuna)
arch_name=regnet_y_16gf tissue_type=palpebral
learning_rate=0.006464900601177843 weight_decay=1.217404417807552e-06 dropout_rate=0.2
MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=15 BATCH_SIZE=32
Pool (train+val): 184 patients -> 3 folds

regnet_y_16gf_palpebral_new_way_kfold3 | Fold 1/3 -- train=122 val=62
[regnet_y_16gf_palpebral_new_way_kfold3 | fold 1] train n=186 (pos=92 neg=94 pos_weight=1.022) val n=62
[regnet_y_16gf_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.1429 -> saved regnet_y_16gf_palpebral_new_way_kfold3_fold1_best.pth
[regnet_y_16gf_palpebral_new_way_kfold3 | fold 1] Epoch   1/250 - train_loss=0.7277 val_loss=0.6494 val_f1=0.1429 lr=6.46e-03
[regnet_y_16gf_palpebral_new_way_kfold3 | fold 1] Epoch   2/250 - train_loss=0.6312 val_loss=1.1399 val_f1=0.0000 lr=6.46e-03
[regnet_y_16gf_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.4118 -> saved regnet_y_16gf_p

In [13]:
# new_way kfold3 4/6 -- convnext_base, palpebral
!python classification/new_way/kfold/train_kfold_convnext_base_palpebral_new_way.py
sync_outputs()

Using device: cuda
Model: convnext_base_palpebral_new_way_kfold3 (fixed-hyperparameter 3-fold CV, no Optuna)
arch_name=convnext_base tissue_type=palpebral
learning_rate=0.0001235308191492332 weight_decay=1.0422971466648463e-06 dropout_rate=0.5
MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=15 BATCH_SIZE=32
Pool (train+val): 184 patients -> 3 folds

convnext_base_palpebral_new_way_kfold3 | Fold 1/3 -- train=122 val=62
[convnext_base_palpebral_new_way_kfold3 | fold 1] train n=186 (pos=92 neg=94 pos_weight=1.022) val n=62
[convnext_base_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.3636 -> saved convnext_base_palpebral_new_way_kfold3_fold1_best.pth
[convnext_base_palpebral_new_way_kfold3 | fold 1] Epoch   1/250 - train_loss=0.7125 val_loss=0.7234 val_f1=0.3636 lr=1.24e-04
[convnext_base_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.3871 -> saved convnext_base_palpebral_new_way_kfold3_fold1_best.pth
[convnext_base_palpebral_new_way_kfold3 | fold 1] Epoch   2/250 - train_loss=0.6978 v

In [14]:
# new_way kfold3 5/6 -- coatnet_3, palpebral
!python classification/new_way/kfold/train_kfold_coatnet_3_palpebral_new_way.py
sync_outputs()

Using device: cuda
Model: coatnet_3_palpebral_new_way_kfold3 (fixed-hyperparameter 3-fold CV, no Optuna)
arch_name=coatnet_3 tissue_type=palpebral
learning_rate=0.0011249436130085291 weight_decay=0.00011856737112037188 dropout_rate=0.2
MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=15 BATCH_SIZE=32
Pool (train+val): 184 patients -> 3 folds

coatnet_3_palpebral_new_way_kfold3 | Fold 1/3 -- train=122 val=62
[coatnet_3_palpebral_new_way_kfold3 | fold 1] train n=186 (pos=92 neg=94 pos_weight=1.022) val n=62
[coatnet_3_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.4058 -> saved coatnet_3_palpebral_new_way_kfold3_fold1_best.pth
[coatnet_3_palpebral_new_way_kfold3 | fold 1] Epoch   1/250 - train_loss=0.8706 val_loss=0.8538 val_f1=0.4058 lr=1.12e-03
[coatnet_3_palpebral_new_way_kfold3 | fold 1] Epoch   2/250 - train_loss=0.6094 val_loss=1.0332 val_f1=0.3333 lr=1.12e-03
[coatnet_3_palpebral_new_way_kfold3 | fold 1] Epoch   3/250 - train_loss=0.5631 val_loss=1.1503 val_f1=0.3881 lr=1.12e-03
[coat

In [15]:
# new_way kfold3 6/6 -- convnext_large, palpebral (heaviest)
!python classification/new_way/kfold/train_kfold_convnext_large_palpebral_new_way.py
sync_outputs()

Using device: cuda
Model: convnext_large_palpebral_new_way_kfold3 (fixed-hyperparameter 3-fold CV, no Optuna)
arch_name=convnext_large tissue_type=palpebral
learning_rate=0.00011137414908293505 weight_decay=2.2059282146979313e-05 dropout_rate=0.2
MAX_EPOCHS=250 EARLY_STOPPING_PATIENCE=15 BATCH_SIZE=32
Pool (train+val): 184 patients -> 3 folds

convnext_large_palpebral_new_way_kfold3 | Fold 1/3 -- train=122 val=62
[convnext_large_palpebral_new_way_kfold3 | fold 1] train n=186 (pos=92 neg=94 pos_weight=1.022) val n=62
[convnext_large_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.5500 -> saved convnext_large_palpebral_new_way_kfold3_fold1_best.pth
[convnext_large_palpebral_new_way_kfold3 | fold 1] Epoch   1/250 - train_loss=0.6902 val_loss=0.6783 val_f1=0.5500 lr=1.11e-04
[convnext_large_palpebral_new_way_kfold3 | fold 1] New best val_f1=0.6818 -> saved convnext_large_palpebral_new_way_kfold3_fold1_best.pth
[convnext_large_palpebral_new_way_kfold3 | fold 1] Epoch   2/250 - train_lo

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever
combos completed) and zipped to `/kaggle/working/new_way_kfold3_results.zip`. Both are visible in
this notebook version's **Output** tab once you Save Version -> Save & Run All -- download the
zip directly from there, or browse the folder for individual files.

Each combo's `{model_name}_kfold_summary.json` has the `aggregate_across_folds` mean+/-std test
metrics -- that's the headline number this whole effort exists to produce.

In [16]:
print("Final contents of /kaggle/working/outputs:")
for f in sorted(Path("/kaggle/working/outputs").rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to('/kaggle/working/outputs')}  ({f.stat().st_size / 1e6:.2f} MB)")

zip_path = Path("/kaggle/working/new_way_kfold3_results.zip")
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

Final contents of /kaggle/working/outputs:
  checkpoints/coatnet_3_palpebral_new_way_kfold3_fold1_best.pth  (327.52 MB)
  checkpoints/coatnet_3_palpebral_new_way_kfold3_fold2_best.pth  (327.52 MB)
  checkpoints/coatnet_3_palpebral_new_way_kfold3_fold3_best.pth  (327.52 MB)
  checkpoints/convnext_base_palpebral_new_way_kfold3_fold1_best.pth  (175.26 MB)
  checkpoints/convnext_base_palpebral_new_way_kfold3_fold2_best.pth  (175.26 MB)
  checkpoints/convnext_base_palpebral_new_way_kfold3_fold3_best.pth  (175.26 MB)
  checkpoints/convnext_large_palpebral_new_way_kfold3_fold1_best.pth  (392.59 MB)
  checkpoints/convnext_large_palpebral_new_way_kfold3_fold2_best.pth  (392.59 MB)
  checkpoints/convnext_large_palpebral_new_way_kfold3_fold3_best.pth  (392.59 MB)
  checkpoints/efficientnet_b3_forniceal_palpebral_new_way_kfold3_fold1_best.pth  (21.79 MB)
  checkpoints/efficientnet_b3_forniceal_palpebral_new_way_kfold3_fold2_best.pth  (21.79 MB)
  checkpoints/efficientnet_b3_forniceal_palpebral_new